In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
import datetime
from keras.datasets import mnist
import wandb

In [15]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNetwork

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
def normalize(x):
    return x.reshape(len(x), -1).astype('float64') / (np.max(x) - np.min(x))

In [17]:
def load_and_prepare_data(dataset="mnist"):
    # Load the Fashion MNIST dataset
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    
    # Using train_test_split to separate validation data (10% of training data)
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=69)
    
    # Normalize image pixel values
    x_train = normalize(x_train)
    x_val   = normalize(x_val)
    x_test  = normalize(x_test)
    
    # Determine the number of classes from the unique labels
    classes = np.unique(y_train)
    num_classes = len(classes)
    
    # One-hot encode labels based on the discovered number of classes
    y_train = np.eye(num_classes)[y_train]
    y_val   = np.eye(num_classes)[y_val]
    y_test  = np.eye(num_classes)[y_test]
    
    return x_train, y_train, x_val, y_val, x_test, y_test

In [18]:
# Initialize WandB API
api = wandb.Api()

# Fetch all runs from the project
project = "fashion-mnist-classification"
runs = api.runs(project)

# Filter finished runs with a valid test_accuracy
valid_runs = [run for run in runs if 'test_accuracy' in run.summary]

# Sort the runs by test_accuracy in descending order and select the top 3
sorted_runs = sorted(valid_runs, key=lambda run: run.summary['test_accuracy'], reverse=True)
top_runs = sorted_runs[:10]

# Print out the top 3 configurations
print("Top 10 Configurations:")
for i, run in enumerate(top_runs):
    config = dict(run.config)
    test_acc = run.summary['test_accuracy']
    print(f"Run {i+1} - Test Accuracy: {test_acc:.4f}")
    print("Configuration:", config)


Top 10 Configurations:
Run 1 - Test Accuracy: 0.8768
Configuration: {'loss': 'mean_squared_error', 'epochs': 10, 'optimizer': 'nag', 'activation': 'ReLU', 'batch_size': 32, 'num_layers': 3, 'hidden_size': 128, 'weight_init': 'Random', 'weight_decay': 0, 'learning_rate': 0.001}
Run 2 - Test Accuracy: 0.8767
Configuration: {'beta': 0.9, 'loss': 'cross_entropy', 'beta1': 0.9, 'beta2': 0.999, 'epochs': 10, 'dataset': 'fashion_mnist', 'epsilon': 1e-06, 'momentum': 0.9, 'optimizer': 'momentum', 'activation': 'Tanh', 'batch_size': 64, 'num_layers': 4, 'hidden_size': 128, 'weight_init': 'Xavier', 'wandb_entity': 'mrsagarbiswas-iit-madras', 'weight_decay': 0.0005, 'learning_rate': 0.0001, 'wandb_project': 'fashion-mnist-classification'}
Run 3 - Test Accuracy: 0.8766
Configuration: {'loss': 'cross_entropy', 'epochs': 10, 'optimizer': 'momentum', 'activation': 'Tanh', 'batch_size': 64, 'num_layers': 4, 'hidden_size': 128, 'weight_init': 'Xavier', 'weight_decay': 0.0005, 'learning_rate': 0.0001}
R

In [19]:
# Load and prepare data
x_train, y_train, x_val, y_val, x_test, y_test = load_and_prepare_data()

top3 = sorted_runs[:3]
# Iterate over the top configurations, rebuild and evaluate the models
for i, run in enumerate(top3):
    config = dict(run.config)
    
    # Initialize a new wandb run for logging re-evaluation metrics
    wandb.init(project="mnist-classification", name=f"re_evaluation_run_{i+1}", reinit=True)
    
    # Rebuild the model using the current configuration
    model = NeuralNetwork(
        input_size=x_train.shape[1],
        num_classes=y_train.shape[1],
        num_hidden=config['num_layers'],
        hidden_units=config['hidden_size'],
        init_method=config['weight_init'],
        activation=config['activation'],
        loss_fn=config['loss'],
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        optimizer=config['optimizer'],
        lr=config['learning_rate'],
        weight_decay=config.get('weight_decay', 0),
        momentum=config.get('momentum', 0.9),
        beta=config.get('beta', 0.9),
        beta1=config.get('beta1', 0.9),
        beta2=config.get('beta2', 0.999),
        epsilon=config.get('epsilon', 1e-6)
    )
    
    # Train the model
    model.fit(x_train, y_train, x_val, y_val)
    
    # Evaluate on the validation set
    val_preds = model.predict(x_val.T)
    val_loss  = model.compute_loss(val_preds, y_val)
    val_acc   = model.accuracy(val_preds, y_val)
    
    # Evaluate on the test set
    test_preds = model.predict(x_test.T)
    test_loss  = model.compute_loss(test_preds, y_test)
    test_acc   = model.accuracy(test_preds, y_test)
    
    # Log evaluation metrics with a timestamp
    wandb.log({
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "created": datetime.datetime.now().isoformat()
    })
    
    wandb.finish()

Epoch 1: train_loss = 0.07, valid_loss = 0.08, train_accuracy = 0.95, val_accuracy = 0.95
Epoch 2: train_loss = 0.05, valid_loss = 0.07, train_accuracy = 0.97, val_accuracy = 0.96
Epoch 3: train_loss = 0.04, valid_loss = 0.06, train_accuracy = 0.97, val_accuracy = 0.96
Epoch 4: train_loss = 0.03, valid_loss = 0.05, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 5: train_loss = 0.03, valid_loss = 0.05, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 6: train_loss = 0.02, valid_loss = 0.05, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 7: train_loss = 0.02, valid_loss = 0.05, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 8: train_loss = 0.02, valid_loss = 0.05, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 9: train_loss = 0.02, valid_loss = 0.05, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 10: train_loss = 0.02, valid_loss = 0.04, train_accuracy = 0.99, val_accuracy = 0.97


test_accuracy,▁
test_loss,▁
val_accuracy,▁
val_loss,▁
created,2025-03-17T01:25:06....
test_accuracy,0.9755
test_loss,0.03855
val_accuracy,0.97217
val_loss,0.04396


Epoch 1: train_loss = 0.24, valid_loss = 0.26, train_accuracy = 0.93, val_accuracy = 0.92
Epoch 2: train_loss = 0.17, valid_loss = 0.19, train_accuracy = 0.95, val_accuracy = 0.95
Epoch 3: train_loss = 0.12, valid_loss = 0.16, train_accuracy = 0.96, val_accuracy = 0.96
Epoch 4: train_loss = 0.10, valid_loss = 0.14, train_accuracy = 0.97, val_accuracy = 0.96
Epoch 5: train_loss = 0.08, valid_loss = 0.13, train_accuracy = 0.98, val_accuracy = 0.96
Epoch 6: train_loss = 0.07, valid_loss = 0.12, train_accuracy = 0.98, val_accuracy = 0.96
Epoch 7: train_loss = 0.06, valid_loss = 0.11, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 8: train_loss = 0.05, valid_loss = 0.11, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 9: train_loss = 0.04, valid_loss = 0.11, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 10: train_loss = 0.04, valid_loss = 0.11, train_accuracy = 0.99, val_accuracy = 0.97


test_accuracy,▁
test_loss,▁
val_accuracy,▁
val_loss,▁
created,2025-03-17T01:25:56....
test_accuracy,0.972
test_loss,0.08427
val_accuracy,0.97067
val_loss,0.10568


Epoch 1: train_loss = 0.24, valid_loss = 0.25, train_accuracy = 0.93, val_accuracy = 0.92
Epoch 2: train_loss = 0.17, valid_loss = 0.19, train_accuracy = 0.95, val_accuracy = 0.94
Epoch 3: train_loss = 0.13, valid_loss = 0.16, train_accuracy = 0.96, val_accuracy = 0.95
Epoch 4: train_loss = 0.10, valid_loss = 0.13, train_accuracy = 0.97, val_accuracy = 0.96
Epoch 5: train_loss = 0.08, valid_loss = 0.12, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 6: train_loss = 0.07, valid_loss = 0.11, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 7: train_loss = 0.06, valid_loss = 0.11, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 8: train_loss = 0.05, valid_loss = 0.10, train_accuracy = 0.98, val_accuracy = 0.97
Epoch 9: train_loss = 0.04, valid_loss = 0.10, train_accuracy = 0.99, val_accuracy = 0.97
Epoch 10: train_loss = 0.04, valid_loss = 0.10, train_accuracy = 0.99, val_accuracy = 0.97


test_accuracy,▁
test_loss,▁
val_accuracy,▁
val_loss,▁
created,2025-03-17T01:26:44....
test_accuracy,0.9732
test_loss,0.08601
val_accuracy,0.96867
val_loss,0.10156
